# Hands-on Part 3: Networks 2 🕸️

In this notebook you will learn:

1. **if conditions**: let your code make decisions,
2. **for loops**: repeat code for every item,
3. **pandas**: load data, clean it, and add information with a dictionary,
4. **directed networks**: followers and following,
5. **community detection**: different algorithms and **modularity**.

As before: run cells from top to bottom with **Shift + Enter**. Cells marked ⚠️ give an error on purpose.

## 0. Setup

We import the libraries and rebuild the friendship network `G` from Part 2.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

friendships = [
    ("Ana", "Ben"), ("Ana", "Chen"), ("Ana", "Dara"), ("Ana", "Eva"),
    ("Ben", "Chen"), ("Ben", "Dara"), ("Chen", "Dara"), ("Dara", "Eva"),
    ("Dara", "Felix"), ("Chen", "Felix"), ("Felix", "Gus"), ("Felix", "Hana"),
    ("Gus", "Hana"), ("Gus", "Ivo"), ("Gus", "Jonas"), ("Gus", "Kim"),
    ("Gus", "Lena"), ("Hana", "Ivo"), ("Kim", "Lena"), ("Ivo", "Jonas"),
    ("Hana", "Kim"),
]

G = nx.Graph()
G.add_edges_from(friendships)

A drawing function that can **colour nodes by group**. **You don't need to understand the code**, only how to use it:

- `draw_network(G)` draws the network.
- `draw_network(G, groups)` colours the nodes. `groups` can be a dictionary `{node: group}` or a list of communities.

In [ ]:
def draw_network(G, groups=None, title=""):
    if isinstance(groups, list):   # a list of communities -> {node: community number}
        groups = {node: number for number, community in enumerate(groups) for node in community}
    palette = ["tab:blue", "tab:red", "tab:green", "tab:orange", "tab:purple"]
    if groups is None:
        colors = "lightblue"
    else:
        labels = sorted(set(groups.values()), key=str)
        colors = [palette[labels.index(groups[node]) % len(palette)] for node in G.nodes]
    positions = nx.kamada_kawai_layout(G.to_undirected())
    plt.figure(figsize=(7, 4))
    nx.draw(G, positions, with_labels=True, node_size=600, node_color=colors,
            edge_color="gray", font_size=9)
    plt.title(title)
    plt.show()

draw_network(G, title="Friend groups")

## 1. If conditions

An **if condition** runs code only when a condition is `True`.

In [ ]:
followers = 1000

if followers >= 100:
    print("This is a popular account.")

- `followers >= 100` is the **condition**: it is either `True` or `False`.
- The **print** line runs only if the condition is `True`.

*Change `followers` to `5` and run the cell again. What happens?*

Conditions use comparisons:

| Operator | Meaning |
|---|---|
| `==` | equal to ⚠️ two `=` signs! |
| `!=` | not equal to |
| `>` / `<` | greater / less than |
| `>=` / `<=` | greater / less than or equal |

### if, elif, else

`else` runs when the condition is `False`. `elif` ("else if") adds more conditions, checked from top to bottom:

In [ ]:
followers = 6

if followers >= 100:
    print("popular")
elif followers >= 5:
    print("average")
else:
    print("few followers")

### and, or

- `and`: **both** conditions must be `True`,
- `or`: **at least one** condition must be `True`.

In [ ]:
followers = 6
following = 40

if followers < 10 and following > 30:
    print("Follows many accounts, but has few followers.")

if followers > 100 or following > 100:
    print("A very active account.")
else:
    print("Not a very active account.")

🕸️ **Conditions on a network:** `G.has_edge(a, b)` is `True` if `a` and `b` are linked.

In [ ]:
if G.has_edge("Felix", "Gus"):
    print("Felix and Gus are friends.")
else:
    print("Felix and Gus are not friends.")

⚠️ **Error on purpose.** One `=` instead of `==`. *What does the error message suggest?*

In [ ]:
if followers = 6:
    print("six followers")

**Task 1:** Create a variable `party = "Green"`. Write an if/elif/else that prints `"Blue party"`, `"Red party"` or `"another party"`. Try different values.

In [ ]:
# your code here

## 2. For loops

A **for loop** runs the same code **for each item** in a list.

In [ ]:
names = ["Ana", "Ben", "Chen"]

for name in names:
    print("Hello,", name)

`name` takes each item of the list in turn. The **indented** code runs once per item.

🕸️ We can loop over the nodes of a network:

In [ ]:
for node in G.nodes:
    print(node, "has", G.degree(node), "friends")

**Loop + if**: only act on some items.

In [ ]:
for node in G.nodes:
    if G.degree(node) >= 4:
        print(node, "is well connected")

**Counting**: start at 0 and add 1 each time the condition is `True`.

In [ ]:
count = 0

for node in G.nodes:
    if G.degree(node) >= 4:
        count = count + 1

print(count, "people have 4 or more friends")

**Task 2:** Count how many people in `G` have **exactly 2** friends.

In [ ]:
# your code here

## 3. pandas: load, clean and add information

Network data usually comes as two files:

- an **edge list**: who follows whom,
- a **node list**: information about each account (here: their party).

The two cells below create these files (look at the 📁 folder on the left afterwards). Each line `source,target` means: **source follows target**.

In [ ]:
%%writefile follows.csv
source,target
Anna,Clara
Ben,Clara
David,Clara
Clara,Anna
Anna,Ben
Ben,David
David,Anna
Emma,Finn
Finn,Greta
Greta,Emma
Finn,Emma
Hugo,Emma
Emma,Hugo
Hugo,Clara
Greta,Clara
Anna,Hugo
Anna,Clara
Ben,

In [ ]:
%%writefile accounts.csv
account,party
Anna,Blue
Ben,Blue
Clara,Blue
David,Blue
Emma,Red
Finn,Red
Greta,Red
Hugo,Red

### Load the data

`pd.read_csv()` loads a file into a table (DataFrame):

In [ ]:
follows = pd.read_csv("follows.csv")
accounts = pd.read_csv("accounts.csv")

follows

### Clean the data

Look at the last row: something is wrong.

**1. Missing values.** `NaN` means a value is missing. `dropna()` removes rows with missing values:

In [ ]:
follows = follows.dropna()
follows.shape   # (rows, columns)

**2. Duplicates.** "Anna, Clara" appears twice. `drop_duplicates()` keeps only one copy of each row:

In [ ]:
follows = follows.drop_duplicates()
follows.shape

### Add information with a dictionary

We make a dictionary `{account: party}` from the accounts table. `zip()` pairs the two columns row by row:

In [ ]:
party_of = dict(zip(accounts["account"], accounts["party"]))
party_of

In [ ]:
party_of["Hugo"]

`map()` looks up every value of a column in the dictionary. This adds the party of each follower and each followed account:

In [ ]:
follows["source_party"] = follows["source"].map(party_of)
follows["target_party"] = follows["target"].map(party_of)
follows

**Task 3:** How many rows did we remove while cleaning? Which follows go **across** parties (source party ≠ target party)?

**Your answer:**

## 4. Directed networks

Following is **directed**: Anna following Clara does **not** mean Clara follows Anna (Week 1). So we build a `DiGraph` from the table:

In [ ]:
F = nx.from_pandas_edgelist(follows, source="source", target="target",
                            create_using=nx.DiGraph())

print("N =", F.number_of_nodes())
print("L =", F.number_of_edges())

draw_network(F, party_of, title="Follower network (blue = Blue party, red = Red party)")

Every node now has two degrees:

- **in-degree**: arrows pointing **to** it = **followers**,
- **out-degree**: arrows pointing **from** it = accounts it **follows**.

In [ ]:
print("Clara has", F.in_degree("Clara"), "followers")
print("Clara follows", F.out_degree("Clara"), "accounts")

🕸️ **Everything together:** a loop over all accounts, the dictionary for the party, and an if condition:

In [ ]:
for account in F.nodes:
    party = party_of[account]
    followers = F.in_degree(account)
    if followers >= 3:
        print(account, "(" + party + ") is popular with", followers, "followers")
    else:
        print(account, "(" + party + ") has", followers, "followers")

**Task 4:** Write a loop that prints every account that **follows more accounts than it has followers** (out-degree > in-degree).

In [ ]:
# your code here

## 5. Community detection

From last week's lecture by Felix: a **community** is a group of nodes with **many links inside** and **few links to the rest** of the network. We do not see communities directly: an **algorithm** finds them.

**Modularity Q** measures how good a split into communities is: **the higher Q, the clearer the communities** (Q = 0 means no better than chance).

We use the friendship network `G` from Part 2 and try **three algorithms**:

| Algorithm | Idea |
|---|---|
| **Louvain** | move nodes between communities as long as Q increases (Gephi uses this) |
| **Label propagation** | every node adopts the most common label among its friends |
| **Girvan–Newman** | keep cutting the link with the highest betweenness (the bridges) |

### Louvain

In [ ]:
louvain = nx.community.louvain_communities(G, seed=1)   # seed=1: same result every time
louvain

The result is a list of communities. Each community is a **set** of nodes (curly brackets `{ }`).

In [ ]:
print("Number of communities:", len(louvain))
print("Modularity:", round(nx.community.modularity(G, louvain), 2))

draw_network(G, louvain, title="Louvain")

### The other algorithms

In [ ]:
label_propagation = list(nx.community.label_propagation_communities(G))
draw_network(G, label_propagation, title="Label propagation")

In [ ]:
girvan_newman = list(next(nx.community.girvan_newman(G)))   # next() gives the first split
draw_network(G, girvan_newman, title="Girvan-Newman")

### Comparing the results

A loop over a dictionary `{name: communities}` gives the number of communities and the modularity for each algorithm:

In [ ]:
results = {
    "Louvain": louvain,
    "Label propagation": label_propagation,
    "Girvan-Newman": girvan_newman,
}

for name, communities in results.items():
    Q = nx.community.modularity(G, communities)
    print(name, ":", len(communities), "communities, modularity =", round(Q, 2))

**Task 5:** Which person is placed in a different community depending on the algorithm? Why is this person hard to place? Which result has the highest modularity?

**Your answer:**

💡 Different algorithms can give different communities for the **same** network. A community is something we **infer**, so always report which algorithm you used.

# Reflection

| Task | Code |
|---|---|
| make decisions | `if`, `elif`, `else`, `==`, `!=`, `and`, `or` |
| repeat for every item | `for node in G.nodes:` |
| load data | `pd.read_csv("file.csv")` |
| clean data | `dropna()`, `drop_duplicates()` |
| add information | `dict(zip(...))`, `df["column"].map(...)` |
| directed network | `nx.from_pandas_edgelist(df, ..., create_using=nx.DiGraph())` |
| followers / following | `F.in_degree(node)`, `F.out_degree(node)` |
| communities | `nx.community.louvain_communities(G, seed=1)` and others |
| modularity | `nx.community.modularity(G, communities)` |